In [57]:
import geopandas as gpd
import pandas as pd
import yaml
from pathlib import Path
from tqdm.auto import tqdm
from rasterio.crs import CRS

# Events

In [42]:
event_ymls = list(Path('db/events/').glob('*.yml'))
event_ymls[:2]

[PosixPath('db/events/chiapas_fire_2024.yml'),
 PosixPath('db/events/yajiang_fire_2024.yml')]

In [47]:
KEYS = ['event_name',
        'event_date',
        'mgrs_tiles',
        'source_id',
        'links']
def open_one_event(path: str) -> dict:
    with open(path) as file:
        nested_data = yaml.safe_load(file)
    data = nested_data['event']
    out = {}
    keys_not_present = [key for key in KEYS if key not in data]
    if keys_not_present:
        print(f'{path} does not have {", ".join(keys_not_present)}')
        print('skipping...')
        return {}
    out = {key: data[key] for key in KEYS}
    return out

In [48]:
event_records = list(map(open_one_event, event_ymls))
event_records[0]

{'event_name': 'chiapas_fire_2024',
 'event_date': '2024-03-24',
 'mgrs_tiles': ['15QUU'],
 'source_id': 'Copernicus EMSR717',
 'links': ['data: https://rapidmapping.emergency.copernicus.eu/EMSR717',
  'https://earthobservatory.nasa.gov/images/152628/fire-in-southern-mexico',
  'date of start: https://emergency.copernicus.eu/mapping/list-of-components/EMSR506',
  'https://www.latimes.com/california/story/2024-03-15/wildfire-devastates-oaxaca-region-famed-for-mezcal',
  'wildfires typicall start in March but actually came early - lots of pre-event dates show signs of burn scarring']}

In [49]:
df_event = pd.DataFrame(event_records)
df_event.head()

,event_name,event_date,mgrs_tiles,source_id,links
0,chiapas_fire_2024,2024-03-24,[15QUU],Copernicus EMSR717,[data: https://rapidmapping.emergency.copernic...
1,yajiang_fire_2024,2024-03-15,"[47RQP, 47RPP]",UNOSAT via humanitarian data exchange,[https://data.humdata.org/dataset/the-wildfire...
2,bangladesh_coastal_flood_2024,2024-07-08,"[45QYE, 45QYF, 45QZE, 45QZF]",UNOSAT via humanitarian data exchange ST1_2024...,[https://data.humdata.org/dataset/satellite-de...
3,park_fire_2024,2024-07-24,"[10TEK, 10TFK]",WFIGS Park Fire 2024,[https://en.wikipedia.org/wiki/Park_Fire]
4,chilcotin_river_landslide_and_flood_2024,2024-07-30,[10UEC],Al Handwerger derived landslide and flood from...,[https://chilcotin-river-landslide-2024-bcgov0...


In [83]:
df_event.to_parquet('event_metadata.parquet')

# Event Perimeters

In [74]:
def open_one_perimeter(event_name: str):
    perimeter_dir = Path('db/event_perimeters/')
    path_geojson = perimeter_dir / f'{event_name}.geojson'
    if not path_geojson.exists():
        print(f'{path_geojson} does not exist; searching for parquet')
        path_parquet = perimeter_dir / f'{event_name}.parquet'
        if not path_parquet.exists():
            print(f'{path_parquet} does not exist')
            print('skipping...')
            return {}
        df_geo = gpd.read_parquet(path_parquet)
    else:
        df_geo = gpd.read_file(path_geojson)
    if df_geo.crs != CRS.from_epsg(4326):
        print(f'CRS for {event_name} is not lon/lat')
        print(f'Please reproject; skipping...')
        return {}
    geo = df_geo.geometry.union_all()
    return {'event_name': event_name, 'geometry': geo}

In [55]:
event_names = df_event['event_name'].tolist()
perimeter_data = [open_one_perimeter(event) for event in tqdm(event_names)]

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:12: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union


db/event_perimeters/bangladesh_coastal_flood_2024.geojson does not exist; searching for parquet


/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:12: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union


db/event_perimeters/demak_flood_2024.geojson does not exist; searching for parquet


/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union'

db/event_perimeters/mai_mahiu_flood_and_landslides_2024.geojson does not exist; searching for parquet


/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo = df_geo.geometry.unary_union
/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/3080771691.py:15: DeprecationWarning: The 'unary_union'

In [59]:
df_perimeters = gpd.GeoDataFrame(perimeter_data,
                                 crs=CRS.from_epsg(4326))
df_perimeters.head()

,event_name,geometry
0,chiapas_fire_2024,"MULTIPOLYGON (((-94.53702 16.47015, -94.53692 ..."
1,yajiang_fire_2024,"MULTIPOLYGON (((101.0631 30.05153, 101.06069 3..."
2,bangladesh_coastal_flood_2024,GEOMETRYCOLLECTION (POLYGON ((88.98224 22.4023...
3,park_fire_2024,"MULTIPOLYGON (((-121.93048 39.97492, -121.9296..."
4,chilcotin_river_landslide_and_flood_2024,"MULTIPOLYGON (((-122.7273 51.84427, -122.72799..."


In [89]:
df_perimeters.to_parquet('event_perimeters.parquet', compression='zstd')

# Event Sites

In [75]:
df_event_sites = df_perimeters.copy()
df_event_sites.geometry = df_perimeters.geometry.centroid

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_28994/2105504291.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_event_sites.geometry = df_perimeters.geometry.centroid


In [76]:
df_event_sites.to_parquet('event_sites.parquet')

# Event Extents

In [77]:
def open_one_extent(event_name: str):
    extent_dir = Path('db/event_extents/')
    path_geojson = extent_dir / f'{event_name}.geojson'
    if not path_geojson.exists():
        print(f'{path_parquet} does not exist')
        print('skipping...')
        return {}
    
    df_geo = gpd.read_file(path_geojson)
    if df_geo.crs != CRS.from_epsg(4326):
        print(f'CRS for {event_name} is not lon/lat')
        print(f'Please reproject; skipping...')
        return {}
    geo = df_geo.geometry.union_all()
    return {'event_name': event_name, 'geometry': geo}

In [78]:
event_names = df_event['event_name'].tolist()
extent_data = [open_one_extent(event) for event in tqdm(event_names)]

  0%|          | 0/21 [00:00<?, ?it/s]

/Users/cmarshak/miniforge3/envs/dist-s1-val/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: Several features with id = 1 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [80]:
df_extent = gpd.GeoDataFrame(extent_data, crs=CRS.from_epsg(4326))
df_extent.to_parquet('event_extents.parquet')

In [ ]:
# DIST-HLS Tiffs